#### DESCARGA DE IMÁGENES DESDE MODIS Y ERA5

### 1. Conexión a GEE

In [2]:
# Verificamos que estamos en el entorno creado para la descarga de datos
import sys
print(sys.executable)

/opt/anaconda3/envs/seguro_cafe/bin/python


In [3]:
# Inicializamos GEE
import ee

ee.Reset()
ee.Initialize(project='redcafe')
print(ee.String('Conexión exitosa con proyecto: redcafe').getInfo())

/opt/anaconda3/envs/seguro_cafe/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


Conexión exitosa con proyecto: redcafe


In [4]:
# Verificamos bandas disponibles en ERA5-Land Daily
imagen = ee.Image('ECMWF/ERA5_LAND/DAILY_AGGR/20200101')
print(imagen.bandNames().getInfo())

['dewpoint_temperature_2m', 'temperature_2m', 'skin_temperature', 'soil_temperature_level_1', 'soil_temperature_level_2', 'soil_temperature_level_3', 'soil_temperature_level_4', 'lake_bottom_temperature', 'lake_ice_depth', 'lake_ice_temperature', 'lake_mix_layer_depth', 'lake_mix_layer_temperature', 'lake_shape_factor', 'lake_total_layer_temperature', 'snow_albedo', 'snow_cover', 'snow_density', 'snow_depth', 'snow_depth_water_equivalent', 'snowfall_sum', 'snowmelt_sum', 'temperature_of_snow_layer', 'skin_reservoir_content', 'volumetric_soil_water_layer_1', 'volumetric_soil_water_layer_2', 'volumetric_soil_water_layer_3', 'volumetric_soil_water_layer_4', 'forecast_albedo', 'surface_latent_heat_flux_sum', 'surface_net_solar_radiation_sum', 'surface_net_thermal_radiation_sum', 'surface_sensible_heat_flux_sum', 'surface_solar_radiation_downwards_sum', 'surface_thermal_radiation_downwards_sum', 'evaporation_from_bare_soil_sum', 'evaporation_from_open_water_surfaces_excluding_oceans_sum', '

### 2. Definición de resolución temporal y areal

In [5]:
# Importamos librerías y definimos región
import ee
import pandas as pd

# Región de Caldas
caldas = ee.Geometry.Rectangle([-76.2, 4.7, -74.9, 5.7])

print('Región definida')

Región definida


In [6]:
# Generar lista de fechas de inicio cada 16 días desde 2003 hasta hoy
fechas = []
fecha  = pd.Timestamp('2003-01-01')
fin    = pd.Timestamp('2026-04-17')

while fecha < fin:
    fechas.append(fecha.strftime('%Y-%m-%d'))
    fecha += pd.Timedelta(days=16)

print(f'Total períodos de 16 días: {len(fechas)}')
print(f'   Primer período: {fechas[0]}')
print(f'   Último período: {fechas[-1]}')

Total períodos de 16 días: 532
   Primer período: 2003-01-01
   Último período: 2026-04-06


### 3. Descarga de datos de ERA5 

In [7]:
# FUNCIÓN imagen_16dias  —  versión final con PET y todas las correcciones

# BANDAS EN EL ARCHIVO DE SALIDA (11 total):
#   tmax          [°C]       máximo de los máximos diarios
#   tmin          [°C]       mínimo de los mínimos diarios
#   skin_temp     [°C]       temperatura superficial del suelo, promedio
#   ppt           [m]        precipitación total acumulada
#   srad          [J/m²]     radiación solar descendente acumulada
#   pet           [mm]       evapotranspiración potencial acumulada (PM FAO-56)
#   soil_water    [m³/m³]    humedad volumétrica suelo 0-7cm, promedio
#   vap           [kPa]      presión de vapor real, promedio
#   vpd           [kPa]      déficit de presión de vapor, promedio
#   ws            [m/s]      velocidad del viento a 10m, promedio de magnitudes
#   surface_pressure [Pa]    presión superficial, promedio


def imagen_16dias(fecha_str):
    start = ee.Date(fecha_str)
    end   = start.advance(16, 'day')

    # Colección diaria sin filtro de bandas: cada sub-función
    col = (ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')
             .filterDate(start, end)
             .filterBounds(caldas))

    # FLUJOS ACUMULATIVOS → SUMA de los 16 días
    # ppt: precipitación total [m/16d]
    ppt  = col.select('total_precipitation_sum').sum().rename('ppt')

    # srad: radiación solar descendente [J/m²/16d]
    srad = col.select('surface_solar_radiation_downwards_sum').sum().rename('srad')

    
    # VARIABLES DE ESTADO → PROMEDIO de los 16 días
    # skin_temperature [K → °C]
    ts_c = (col.select('skin_temperature')
               .map(lambda img: img.subtract(273.15).rename('skin_temp'))
               .mean())

    # Humedad volumétrica del suelo [m³/m³]
    soil = col.select('volumetric_soil_water_layer_1').mean().rename('soil_water')

    # Presión superficial [Pa]
    sp   = col.select('surface_pressure').mean().rename('surface_pressure')


    # tmin y tmax
    tmin = (col.select('temperature_2m_min')
               .map(lambda img: img.subtract(273.15).rename('tmin'))
               .min())   # mínimo de los mínimos diarios

    tmax = (col.select('temperature_2m_max')
               .map(lambda img: img.subtract(273.15).rename('tmax'))
               .max())   # máximo de los máximos diarios


    # ws — calculo de sqrt(u²+v²) por día y luego promediar
    def calcular_ws_diario(img):
        u = img.select('u_component_of_wind_10m')
        v = img.select('v_component_of_wind_10m')
        return u.pow(2).add(v.pow(2)).sqrt().rename('ws')

    ws = col.map(calcular_ws_diario).mean()   # [m/s]


    # vap — Magnus por día, luego promedio
    # vap = 0.6108 * exp(17.27 * Td / (Td + 237.3))  [kPa]
    def calcular_vap_diario(img):
        tdew = img.select('dewpoint_temperature_2m').subtract(273.15)
        return (tdew.multiply(17.27)
                    .divide(tdew.add(237.3))
                    .exp()
                    .multiply(0.6108)
                    .rename('vap'))

    vap = col.map(calcular_vap_diario).mean()

    # vpd — por día, luego promedio
    # vpd = vap_sat(Tmean) - vap_real(Tdew)  [kPa], siempre >= 0
    def calcular_vpd_diario(img):
        tmax_d  = img.select('temperature_2m_max').subtract(273.15)
        tmin_d  = img.select('temperature_2m_min').subtract(273.15)
        tdew_d  = img.select('dewpoint_temperature_2m').subtract(273.15)
        tmean_d = tmax_d.add(tmin_d).divide(2)

        vap_sat_d = (tmean_d.multiply(17.27)
                            .divide(tmean_d.add(237.3))
                            .exp()
                            .multiply(0.6108))
        vap_act_d = (tdew_d.multiply(17.27)
                           .divide(tdew_d.add(237.3))
                           .exp()
                           .multiply(0.6108))
        return vap_sat_d.subtract(vap_act_d).rename('vpd')

    vpd = col.map(calcular_vpd_diario).mean()   # [kPa]


    # pet — Penman-Monteith FAO-56 por día → SUMA 16 días
    
    #        0.408·Δ·(Rn-G)  +  γ·[900/(Tmean+273)]·u2·(es-ea)
    # ET0 = ──────────────────────────────────────────────────────
    #                    Δ  +  γ·(1 + 0.34·u2)
    #
    # Conversiones desde ERA5-Land:
    #   Radiación:  J/m²  → MJ/m²  (÷ 1e6)
    #   Temperatura: K    → °C     (- 273.15)
    #   Presión:    Pa    → kPa    (÷ 1000)
    #   Viento 10m → 2m:  u2 = u10 × 0.7481
    def calcular_pet_diario(img):

        # 1. Radiación neta Rn [MJ/m²/día]
        #    Rnl es negativa en ERA5 (pérdida hacia la atmósfera) → suma algebraica
        #    Se fuerza Rn >= 0 (sin sentido físico ET0 con Rn negativa en trópico)
        Rns = img.select('surface_net_solar_radiation_sum').divide(1e6)
        Rnl = img.select('surface_net_thermal_radiation_sum').divide(1e6)
        Rn  = Rns.add(Rnl).max(ee.Image.constant(0))

        # 2. Flujo calor suelo G = 0 (FAO-56, escala diaria)

        # 3. Temperaturas [°C]
        tmax_d  = img.select('temperature_2m_max').subtract(273.15)
        tmin_d  = img.select('temperature_2m_min').subtract(273.15)
        tdew_d  = img.select('dewpoint_temperature_2m').subtract(273.15)
        tmean_d = tmax_d.add(tmin_d).divide(2)

        # 4. Velocidad del viento a 2m [m/s]
        #    u2 = u10 × 4.87 / ln(67.8×10 - 5.42) = u10 × 0.7481
        u10  = img.select('u_component_of_wind_10m')
        v10  = img.select('v_component_of_wind_10m')
        ws10 = u10.pow(2).add(v10.pow(2)).sqrt()
        u2   = ws10.multiply(0.7481)

        # 5. Presión de vapor de saturación es [kPa]
        #    FAO-56: es = (e(Tmax) + e(Tmin)) / 2
        def e_sat(T):
            return T.multiply(17.27).divide(T.add(237.3)).exp().multiply(0.6108)

        es = e_sat(tmax_d).add(e_sat(tmin_d)).divide(2)

        # 6. Presión de vapor real ea [kPa]
        #    Mejor estimador FAO-56: ea = e(Tdew)
        ea = e_sat(tdew_d)

        # 7. Pendiente curva presión de vapor Δ [kPa/°C]
        #    Δ = 4098 × e(Tmean) / (Tmean + 237.3)²
        delta = e_sat(tmean_d).multiply(4098).divide(tmean_d.add(237.3).pow(2))

        # 8. Constante psicrométrica γ [kPa/°C]
        #    γ = 0.000665 × P [kPa]
        P_kPa = img.select('surface_pressure').divide(1000)
        gamma  = P_kPa.multiply(0.000665)

        # 9. Penman-Monteith FAO-56 [mm/día]
        num_rad  = delta.multiply(Rn).multiply(0.408)
        num_aero = (gamma
                    .multiply(ee.Image.constant(900).divide(tmean_d.add(273)))
                    .multiply(u2)
                    .multiply(es.subtract(ea)))
        denominador = delta.add(gamma.multiply(u2.multiply(0.34).add(1)))

        et0 = num_rad.add(num_aero).divide(denominador).max(ee.Image.constant(0))
        return et0.rename('pet')

    # PET es flujo acumulativo → SUMA los 16 días [mm/16d]
    pet = col.map(calcular_pet_diario).sum()


    # Unir todas las bandas
    imagen_final = (tmax
                    .addBands(tmin)
                    .addBands(ts_c)
                    .addBands(ppt)
                    .addBands(srad)
                    .addBands(pet)
                    .addBands(soil)
                    .addBands(vap)
                    .addBands(vpd)
                    .addBands(ws)
                    .addBands(sp)
                    .set('fecha', fecha_str))

    return imagen_final.toFloat()


print('Función definida correctamente')
print('Bandas de salida: tmax, tmin, skin_temp, ppt, srad, pet, soil_water, vap, vpd, ws, surface_pressure')

Función definida correctamente
Bandas de salida: tmax, tmin, skin_temp, ppt, srad, pet, soil_water, vap, vpd, ws, surface_pressure


In [8]:
# VERIFICACIÓN — verificamos que la función que definimos se pueda ejecutar antes de lanzar las 532 tareas
test_img   = imagen_16dias('2010-06-01')
punto_test = ee.Geometry.Point([-75.5, 5.2])   

print('Bandas en la imagen:', test_img.bandNames().getInfo())

valores = test_img.reduceRegion(
    reducer  = ee.Reducer.first(),
    geometry = punto_test,
    scale    = 11132
).getInfo()

print('\nValores de prueba — punto central Caldas, período 2010-06-01:')
for k, v in sorted(valores.items()):
    if v is not None:
        print(f'  {k:30s}: {v:.4f}')
    else:
        print(f'  {k:30s}: None  <- REVISAR')

tmax_v = valores.get('tmax')
tmin_v = valores.get('tmin')
vpd_v  = valores.get('vpd')
pet_v  = valores.get('pet')
ppt_v  = valores.get('ppt')
ws_v   = valores.get('ws')

print('\nValidaciones:')
print(f'  tmax > tmin              : {tmax_v > tmin_v}   (esperado: True)')
print(f'  vpd >= 0                 : {vpd_v >= 0}   (esperado: True)')
print(f'  ppt >= 0                 : {ppt_v >= 0}   (esperado: True)')
print(f'  ws >= 0                  : {ws_v >= 0}   (esperado: True)')
print(f'  tmin en [-5, 35] °C      : {-5 < tmin_v < 35}   (rango físico Caldas)')
print(f'  tmax en [5, 45] °C       : {5 < tmax_v < 45}   (rango físico Caldas)')
print(f'  pet en [20, 150] mm/16d  : {20 < pet_v < 150}   (rango esperado zona tropical)')

print('\nSi todas las validaciones son True, proceder a lanzar las tareas.')

Bandas en la imagen: ['tmax', 'tmin', 'skin_temp', 'ppt', 'srad', 'pet', 'soil_water', 'vap', 'vpd', 'ws', 'surface_pressure']

Valores de prueba — punto central Caldas, período 2010-06-01:
  pet                           : 46.4595
  ppt                           : 0.4425
  skin_temp                     : 16.5217
  soil_water                    : 0.4250
  srad                          : 283706688.0000
  surface_pressure              : 80967.5625
  tmax                          : 24.6861
  tmin                          : 10.4290
  vap                           : 1.6632
  vpd                           : 0.2867
  ws                            : 0.4064

Validaciones:
  tmax > tmin              : True   (esperado: True)
  vpd >= 0                 : True   (esperado: True)
  ppt >= 0                 : True   (esperado: True)
  ws >= 0                  : True   (esperado: True)
  tmin en [-5, 35] °C      : True   (rango físico Caldas)
  tmax en [5, 45] °C       : True   (rango físico Caldas)


### 4. Descarga de datos de MODIS

In [9]:
# DESCARGA MODIS MOD13Q1 — NDVI con corrección espacial a grid ERA5

# MODIS MOD13Q1: composición de 16 días, resolución 250m, desde 2003.

# CORRECCIÓN ESPACIAL:
#   ERA5-Land: ~11km por celda
#   MODIS NDVI: 250m por píxel → ~44×44 = ~1.936 píxeles por celda ERA5

#   Se usa reduceResolution(mean) para agregar los píxeles de 250m al
#   grid de 11km mediante promedio.

#   Se usa reproject(scale=11132) para forzar el mismo grid que ERA5,
#   garantizando que los centroides de celda coincidan exactamente en el
#   join posterior del ETL.

# MÁSCARA DE CALIDAD:
#   SummaryQA = 0: buena calidad
#   SummaryQA = 1: calidad marginal (se acepta)
#   SummaryQA = 2: cubierto por nieve/hielo (se rechaza)
#   SummaryQA = 3: cubierto por nubes (se rechaza)

#   Se agrega SummaryQA al mismo grid 11km (promedio de píxeles de 250m).
#   Celdas donde el promedio de SummaryQA > 1.0 tienen mayoría de píxeles
#   de baja calidad y se enmascaran → equivale al filtro de >=30% faltantes


# BANDAS EN EL ARCHIVO DE SALIDA (1 banda):
#   NDVI   [-1, 1]   promedio de píxeles 250m dentro de la celda 11km,
#                    enmascarado donde calidad promedio > 1.0


def ndvi_16dias(fecha_str):
    start = ee.Date(fecha_str)
    end   = start.advance(16, 'day')

    col_modis = (ee.ImageCollection('MODIS/061/MOD13Q1')
                   .filterDate(start, end)
                   .filterBounds(caldas))

    # MOD13Q1 tiene una sola imagen por período de 16 días, si no hay imagen para este período, retornar imagen vacía enmascarada
    n_imagenes = col_modis.size().getInfo()
    if n_imagenes == 0:
        print(f'  AVISO: sin imagen MODIS para {fecha_str}')
        return None

    imagen_modis = col_modis.first()

    # NDVI: escalar de entero a flotante 
    # MOD13Q1 entrega NDVI como entero × 10000 → dividimos para [-1, 1]
    ndvi_250m = imagen_modis.select('NDVI').multiply(0.0001).rename('NDVI')

    # ── NDVI: corrección espacial 250m → 11km, garantizando el mismo grid de ERA5
    ndvi_11km = (ndvi_250m
                 .reduceResolution(
                     reducer    = ee.Reducer.mean(),
                     bestEffort = True,    
                     maxPixels  = 2048     
                 )
                 .reproject(
                     crs   = 'EPSG:4326',
                     scale = 11132         
                 ))

    #Máscara de calidad: SummaryQA agregado al grid 11km
    qa_250m = imagen_modis.select('SummaryQA')

    qa_11km = (qa_250m
               .reduceResolution(
                   reducer    = ee.Reducer.mean(),
                   bestEffort = True,
                   maxPixels  = 2048
               )
               .reproject(
                   crs   = 'EPSG:4326',
                   scale = 11132
               ))

    # Enmascarar celdas donde el promedio de SummaryQA > 1.0, (más del 50% de píxeles con calidad mala: nieve, nubes)
    mascara_calidad = qa_11km.lte(1.0)
    ndvi_final = ndvi_11km.updateMask(mascara_calidad)

    return ndvi_final.set('fecha', fecha_str).toFloat()


print('Función ndvi_16dias definida correctamente')

Función ndvi_16dias definida correctamente


In [10]:
# VERIFICACIÓN — verificamos que la función que definimos se pueda ejecutar antes de lanzar las 532 tareas
test_ndvi  = ndvi_16dias('2010-06-01')
punto_test = ee.Geometry.Point([-75.5, 5.2])  

print('Bandas en la imagen MODIS:', test_ndvi.bandNames().getInfo())

val_ndvi = test_ndvi.reduceRegion(
    reducer  = ee.Reducer.first(),
    geometry = punto_test,
    scale    = 11132
).getInfo()

print('\nValores de prueba — MODIS NDVI, punto central Caldas, 2010-06-01:')
for k, v in sorted(val_ndvi.items()):
    if v is not None:
        print(f'  {k:10s}: {v:.4f}')
    else:
        print(f'  {k:10s}: None  <- celda enmascarada por baja calidad')

ndvi_v = val_ndvi.get('NDVI')
print('\nValidaciones MODIS:')
print(f'  NDVI en [-1, 1] : {-1 <= ndvi_v <= 1 if ndvi_v is not None else "enmascarada"}')
print(f'  NDVI > 0        : {ndvi_v > 0 if ndvi_v is not None else "enmascarada"}   (esperado en zona cafetera)')

print('\nSi los valores son razonables, proceder a lanzar las tareas MODIS.')

Bandas en la imagen MODIS: ['NDVI']

Valores de prueba — MODIS NDVI, punto central Caldas, 2010-06-01:
  NDVI      : None  <- celda enmascarada por baja calidad

Validaciones MODIS:
  NDVI en [-1, 1] : enmascarada
  NDVI > 0        : enmascarada   (esperado en zona cafetera)

Si los valores son razonables, proceder a lanzar las tareas MODIS.


### 5. Lanzamiento de tareas de MODIS a Google Drive

In [11]:
import time

LOTE = 500

#Registramos los periodos sin imagen disponible
fechas_sin_modis = []  

for i, fecha_str in enumerate(fechas):
    img_ndvi = ndvi_16dias(fecha_str)

    if img_ndvi is None:
        fechas_sin_modis.append(fecha_str)
        continue

    task = ee.batch.Export.image.toDrive(
        image          = img_ndvi,
        description    = f'MODIS_NDVI_Caldas_{fecha_str}',
        folder         = 'MODIS_Caldas',
        fileNamePrefix = f'MODIS_NDVI_Caldas_{fecha_str}',
        region         = caldas,
        scale          = 11132,
        crs            = 'EPSG:4326',
        maxPixels      = 1e9
    )
    task.start()

    if (i + 1) % LOTE == 0:
        print(f'Lote {(i+1)//LOTE} completado ({i+1} tareas). Esperando 60s...')
        time.sleep(60)

    if (i + 1) % 100 == 0:
        print(f'  {i+1} / {len(fechas)} períodos procesados...')

print(f'\nTareas MODIS lanzadas')
if fechas_sin_modis:
    print(f'Períodos sin imagen MODIS ({len(fechas_sin_modis)}): {fechas_sin_modis}')
else:
    print('Todos los períodos tienen imagen MODIS disponible.')
print('Monitorea en: https://code.earthengine.google.com/tasks')

  100 / 532 períodos procesados...
  200 / 532 períodos procesados...
  300 / 532 períodos procesados...
  400 / 532 períodos procesados...
Lote 1 completado (500 tareas). Esperando 60s...
  500 / 532 períodos procesados...
  AVISO: sin imagen MODIS para 2026-04-06

Tareas MODIS lanzadas
Períodos sin imagen MODIS (1): ['2026-04-06']
Monitorea en: https://code.earthengine.google.com/tasks


### 6. Lanzamiento de tareas de ERA5 a Google Drive

In [12]:
import time

#Definimos un lote para no saturar la API de GEE
LOTE = 500   
for i, fecha_str in enumerate(fechas):
    img = imagen_16dias(fecha_str)

    task = ee.batch.Export.image.toDrive(
        image          = img,
        description    = f'ERA5_Caldas_{fecha_str}',
        folder         = 'ERA5_Caldas',
        fileNamePrefix = f'ERA5_Caldas_{fecha_str}',
        region         = caldas,
        scale          = 11132,
        crs            = 'EPSG:4326',
        maxPixels      = 1e9
    )
    task.start()

    if (i + 1) % LOTE == 0:
        print(f'Lote {(i+1)//LOTE} completado ({i+1} tareas). Esperando 60s...')
        time.sleep(60)

    if (i + 1) % 100 == 0:
        print(f'  {i+1} / {len(fechas)} tareas lanzadas...')

print(f'\nTodas las tareas lanzadas ({len(fechas)} períodos)')
print('Monitorea en: https://code.earthengine.google.com/tasks')

  100 / 532 tareas lanzadas...
  200 / 532 tareas lanzadas...
  300 / 532 tareas lanzadas...
  400 / 532 tareas lanzadas...
Lote 1 completado (500 tareas). Esperando 60s...
  500 / 532 tareas lanzadas...

Todas las tareas lanzadas (532 períodos)
Monitorea en: https://code.earthengine.google.com/tasks
